# Perception Eval + Data Engine (Research Notebook Template)

这份 notebook 用于生成**论文式图表**与可复现实验结论。

**目标**：用一个统一的评测/切片/统计框架，把不同 baseline（BEV/3D/occupancy/稀疏）输出变成可比较的证据链。

---

## Checklist（跑之前确认）
- 你有一个 NuScenes 格式数据目录（可用你们 8k 帧 + 公开 nuScenes 叠加）
- 你能拿到至少 1 个 baseline 的预测结果（先离线推理导出即可）
- 你定义了切片字典（距离/遮挡/雨夜/反光/速度/场景段落…）

## 1. Problem Statement
用 3–5 句话写清楚你要解决的可迁移问题：
- 误检（FP taxonomy）
- 远距（distance buckets）
- 稳定性（temporal consistency）
- 校准与可靠性（ECE/NLL，slice-wise calibration）

并写明约束：算力、数据规模、时序/多传感器、部署延迟等。

In [ ]:
# 2. Setup
from __future__ import annotations

import json
import math
import os
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, List, Tuple

import numpy as np
import pandas as pd

import matplotlib as mpl
import matplotlib.pyplot as plt

In [ ]:
# Paper-like plotting style
mpl.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 300,
    'font.size': 11,
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'legend.fontsize': 10,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'lines.linewidth': 2.0,
})

FIG_DIR = Path('reports/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

def savefig(name: str) -> None:
    out_pdf = FIG_DIR / f'{name}.pdf'
    out_png = FIG_DIR / f'{name}.png'
    plt.tight_layout()
    plt.savefig(out_pdf)
    plt.savefig(out_png)
    print('saved:', out_pdf, out_png)

## 3. Data & Protocol
在这里填写你的数据路径与版本号，并做一次**分布体检**：
- 距离分布
- 类别分布
- 场景字典覆盖（固定线路段落 / 雨夜 / 反光 / 遮挡）

建议把体检图作为论文图 1。

In [ ]:
# Set your local paths here
DATA_ROOT = Path(os.environ.get('NUSCENES_ROOT', '/path/to/nuscenes'))
PRED_ROOT = Path(os.environ.get('PRED_ROOT', 'predictions'))

assert DATA_ROOT.exists(), f'DATA_ROOT not found: {DATA_ROOT}'
PRED_ROOT.mkdir(parents=True, exist_ok=True)

print('DATA_ROOT:', DATA_ROOT)
print('PRED_ROOT:', PRED_ROOT)

## 4. Prediction Schema（建议你统一成一个中间格式）
为了可迁移，你最好在这里定义一个**模型无关预测中间格式**，把不同 baseline 的输出先转换成统一 schema，再做评测。

最低限度字段建议：
- `sample_token`
- `class_name`
- `score`（置信度）
- `box`（中心/尺寸/朝向）
- `velocity`（可选）

你可以用 `parquet` 或 `jsonl` 落盘。

In [ ]:
@dataclass(frozen=True)
class Det3D:
    sample_token: str
    class_name: str
    score: float
    x: float
    y: float
    z: float
    w: float
    l: float
    h: float
    yaw: float
    vx: float | None = None
    vy: float | None = None


def load_predictions_json(path: Path) -> pd.DataFrame:
    # Placeholder: adapt to your baseline export format
    with path.open('r', encoding='utf-8') as f:
        obj = json.load(f)
    # Expect a list[dict] for simplicity
    return pd.DataFrame(obj)

# Example: pred_df = load_predictions_json(PRED_ROOT / 'baseline_A.json')

## 5. Slice Definitions
切片建议先做 4 类：
- 距离：0–30 / 30–60 / 60+（按你业务改）
- 遮挡：可用 GT attribute 或近似规则
- 场景字典：固定线路段落（你手工标注一份）
- 环境：雨夜/反光（先用启发式标签也行）

## 1.5 Occ3D-nuScenes Protocol（如果你用 Occ3D）
Occ3D-nuScenes 的常用体素设定（建议写入你的实验协议）：
- voxel size: 0.4m
- range: [-40, -40, -1, 40, 40, 5.4]
- volume: [200, 200, 16]
- classes: 0–17（17 为 free）
- 评测时通常只在 `mask_camera==1` 的体素上计算 mIoU

你后续的所有图表与结论，最好都明确是否使用了可见性 mask。

In [ ]:
def assign_distance_bucket(distance_m: float) -> str:
    if distance_m < 30:
        return '0-30'
    if distance_m < 60:
        return '30-60'
    return '60+'

# You will need a function to compute distance per GT/pred; keep it as a placeholder for now.
def compute_distance_placeholder(x: float, y: float) -> float:
    return float(math.sqrt(x * x + y * y))

## 6. Metrics（先做诊断型，再补主指标）
如果你不想一开始就重实现完整官方指标，可以先做诊断型指标：
- FP rate / FN rate（按类、按距离、按切片）
- 置信度分布（分桶）
- 稳定性（帧间抖动 / track 断裂率，若可得）

这些指标足以支撑作品集叙事；之后再逐步对齐到官方 mAP/NDS。

In [ ]:
def masked_miou(y_true: np.ndarray, y_pred: np.ndarray, mask: np.ndarray, num_classes: int, ignore_index: int | None = None) -> tuple[float, np.ndarray]:
    # y_true/y_pred: int arrays [Z,Y,X]
    # mask: bool/int [Z,Y,X], where True means valid voxel
    assert y_true.shape == y_pred.shape == mask.shape
    valid = mask.astype(bool)
    if ignore_index is not None:
        valid &= (y_true != ignore_index)
    ious = np.full(num_classes, np.nan, dtype=np.float64)
    for c in range(num_classes):
        if ignore_index is not None and c == ignore_index:
            continue
        gt_c = (y_true == c) & valid
        pr_c = (y_pred == c) & valid
        inter = np.sum(gt_c & pr_c)
        union = np.sum(gt_c | pr_c)
        if union == 0:
            continue
        ious[c] = inter / union
    miou = float(np.nanmean(ious))
    return miou, ious